# Low-dimensional Linear Regression (family = 'reg_ld')

In this module, we assume that the conditional outcome model in each source domain $1 \le l \le L$ admits a linear form:

$$
Y^{(l)} = (\theta^{(l)})^\top X^{(l)} + \varepsilon^{(l)}, 
\qquad \text{with} \quad \mathbb{E}[\varepsilon^{(l)}|X^{(l)}] = 0,
$$

which implies
$$
\mathbb{E}[Y^{(l)}|X^{(l)}] = (\theta^{(l)})^\top X^{(l)}.
$$

To learn a **robust prediction model** under domain shifts, the CGDRO framework solves the following minimax optimization problem:

$$
\theta^* = \arg\min_{\theta \in \mathbb{R}^d}
\max_{\mathbf{T} \in \mathcal{C}}
\mathbb{E}_{(X, Y) \sim \mathbf{T}} \ell(X, Y; f_\theta),
\qquad \text{where } f_\theta(X) = \theta^\top X,
$$

where $\mathcal{C}$ is the **uncertainty class** over possible target distributions, as defined in the [Introduction](../setup/intro.md).

We can use `cgdro_()` with `family = 'reg_ld'` for low-dimensional linear regressions.

Now we give an example showing how to implement `family = 'reg_ld'` with three different loss functions:

- Reward-based loss  
- Squared loss  
- Regret-based loss  

---

## Example

### Data Generating Process

In this example, we generate a multi-source domain data with $3$ domains, putting $1,000$ samples on each source domain and $10,000$ samples on the target domain. The dimension of the parameters is $p=5$.

In [ ]:
# number of source groups = 3, with 1000 samples each
# sigma: source group 1,3: 0.5; source group 2: 2
# target sample size = 10000
# dimension p = 5
data <- simu_linear_reg_lowd(n_list = list(1000,1000,1000), N=10000, p = 5, seed = 123)
Xlist = data$X_list
Ylist = data$Y_list
X0 = data$X0

### Implementation & Results

We implement three loss functions by `family = 'reg_ld'`, including `reward`, `squaredloss`, and `regret`. Geometrically, `reward`: $f^∗$ is the point closest to the original within the convex hull of ${f(l)}_{l\in[L]}$; `squaredloss`: $f^{sq}$ corresponds to the source model with the largest noise level with the highest noise level when this noise is substantially higher than that in other sources; `regret`: $f^{reg}$ is the center of the smallest circle enclosing all individual source models.

**Note:** In `Regression.linear.ld`, only when `loss_type`=`reward` can we do inference to get confidence intervals, or we can only do point estimation and prediction.

![Loss Types](../assets/loss_type.png)

#### loss_type =  reward

We define the loss as  
$\ell(X,Y;\theta) = (Y - \theta^\top X)^2 - Y^2$.

Under this choice, the minimax problem becomes

$$
\begin{aligned}
\theta^*
&= \arg\min_{\theta} \max_{\mathbf{T}\in \mathcal{C}}
\mathbb{E}_{\mathbf{T}}\!\left[(Y-\theta^\top X)^2 - Y^2\right] \\[2mm]
&= \arg\max_{\theta} \min_{\mathbf{T}\in \mathcal{C}}
\mathbb{E}_{\mathbf{T}}\!\left[Y^2 - (Y-\theta^\top X)^2\right].
\end{aligned}
$$

The right-hand side shows that  $\mathbb{E}_{\mathbf{T}}\!\left[Y^2 - (Y - \theta^\top X)^2\right]$  can be interpreted as the **explained variance** of $Y$ by the predictor $\theta^\top X$ (assuming $Y$ is centered).  
Hence, the CGDRO model maximizes the **worst-case explained variance** across all possible target distributions in $\mathcal{C}$.


**Proposition**

The CGDRO model $\theta^*$ with reward-based loss admits the closed form:

$$
\theta^* = \sum_{l=1}^L q_l^* \, \theta^{(l)},
\qquad
q^* = \arg\min_{q \in \Delta^L} q^\top \Gamma q,
$$

where $\Gamma \in \mathbb{R}^{L \times L}$ is defined by  

$$
\Gamma_{k,l} = (\theta^{(k)})^\top \mathbb{E}_{\mathbf{Q}}[XX^\top] \, \theta^{(l)}, 
\quad k,l \in [L].
$$

This result implies that $\theta^*$ is a **convex combination** of the source-specific conditional models $\{\theta^{(l)}\}$,  
with weights $q^*$ minimizing the second-moment matrix of the target covariates.

In [ ]:
## Note: only when loss_type='reward', infer_cgdro_() can be called to get confidence intervals
## For other loss_type, only point estimation and prediction can be done

fit <- cgdro_(Xlist, Ylist, X0, loss_type = "reward",
             family = "reg_ld",  intercept = TRUE,
             delta = 0,  verbose = FALSE)
inf <- infer_cgdro_(fit, M = 200, alpha = 0.05)

In [ ]:
summary_cgdro_(fit, infer=inf)

Model Summary:
CGDRO Aggregated Weights:

group     |        1        2        3
weight_   |   0.5523   0.2813   0.1665

CGDRO Aggregated Estimators:

index     |        1        2        3        4        5        6
coef_     |   0.0232  -0.0653  -0.0449   0.0333  -0.0104   0.1307

Confidence Intervals:

index     |              1              2              3              4              5
CI        | (-0.0467,0.1067) (-0.2273,0.0549) (-0.1714,0.0781) (-0.1005,0.1320) (-0.1502,0.1224)
index     |              6
CI        | (0.0247,0.2149)



In [ ]:
pred <- predict_cgdro_(fit)  # N x 1 vector of predicted values
head(pred)

[1] -0.10216265 -0.23954467  0.05219648 -0.14650883 -0.02947897  0.01311665

#### loss_type =  squaredloss

We may also choose the standard squared loss
$\ell(X,Y;\theta) = (Y - \theta^\top X)^2$.

Then the minimax problem becomes

$$
\theta_{\text{sq}} =
\arg\min_{\theta} \max_{\mathbf{T} \in \mathcal{C}}
\mathbb{E}_{(X,Y)\sim \mathbf{T}} (Y - \theta^\top X)^2.
$$

Define the noise level in each source domain:

$$
(\sigma^{(l)})^2 = \mathbb{E}\!\left[(\varepsilon_i^{(l)})^2 \mid X_i^{(l)}\right],
\qquad
\varepsilon_i^{(l)} = Y_i^{(l)} - (\theta^{(l)})^\top X_i^{(l)}.
$$

Let $\boldsymbol{\sigma}^2 = ((\sigma^{(1)})^2, \ldots, (\sigma^{(L)})^2)$.

**Proposition**

The CGDRO model $\theta_{\text{sq}}$ under squared loss admits the closed form:

$$
\theta_{\text{sq}} = \sum_{l=1}^L q_l^{\text{sq}} \, \theta^{(l)},
\qquad
q_{\text{sq}} = \arg\min_{q \in \Delta^L}
q^\top \Gamma q - q^\top (\gamma + \boldsymbol{\sigma}^2),
$$

where $\Gamma \in \mathbb{R}^{L\times L}$ is defined as

$$
\Gamma_{k,l} = (\theta^{(k)})^\top \mathbb{E}_{\mathbf{Q}}[XX^\top] \, \theta^{(l)},
$$

and $\gamma \in \mathbb{R}^L$ is the diagonal of $\Gamma$:
$\gamma_l = \Gamma_{l,l}$ for $l \in [L]$.


In [ ]:
fit <- cgdro_(Xlist, Ylist, X0, loss_type = "squaredloss",
             family = "reg_ld",  intercept = TRUE,
             delta = 0,  verbose = FALSE)

In [ ]:
summary_cgdro_(fit)

Model Summary:
CGDRO Aggregated Weights:

group     |        1        2        3
weight_   |   0.0000   1.0000   0.0000

CGDRO Aggregated Estimators:

index     |        1        2        3        4        5        6
coef_     |   0.0533  -0.4480  -0.2918  -0.2533  -0.2652  -0.1019

Confidence Intervals not provided. Run infer_reg_ld() and pass its result via infer=.


In [ ]:
pred <- predict_cgdro_(fit)  
head(pred)

[1] -0.55637468  0.32233275 -0.68009198  0.62585294 -0.04045642 -0.40336675

#### loss_type =  regret

The **regret** is defined as

$$
\mathrm{Regret}_{\mathbf{T}}(\theta)
:= \mathbb{E}_{\mathbf{T}}\!\left[(Y-\theta^\top X)^2\right]
- \inf_{\theta'} \mathbb{E}_{\mathbf{T}}\!\left[(Y - \theta'^\top X)^2\right].
$$

It measures the **excess risk** of $\theta$ compared to the optimal model for distribution $\mathbf{T}$.  
The CGDRO formulation then seeks to minimize the **worst-case regret**:

$$
\theta_{\text{reg}} =
\arg\min_{\theta}
\max_{\mathbf{T} \in \mathcal{C}}
\mathrm{Regret}_{\mathbf{T}}(\theta).
$$

**Proposition**

The CGDRO model $\theta_{\text{reg}}$ under regret function admits the closed form:

$$
\theta_{\text{reg}} = \sum_{l=1}^L q_l^{\text{reg}} \, \theta^{(l)},
\qquad
q_{\text{reg}} = \arg\min_{q \in \Delta^L}
q^\top \Gamma q - q^\top \gamma,
$$

where $\Gamma \in \mathbb{R}^{L\times L}$ is defined as

$$
\Gamma_{k,l} = (\theta^{(k)})^\top \mathbb{E}_{\mathbf{Q}}[XX^\top] \, \theta^{(l)},
$$

and $\gamma \in \mathbb{R}^L$ is the diagonal of $\Gamma$:
$\gamma_l = \Gamma_{l,l}$ for $l \in [L]$.


In [ ]:
fit <- cgdro_(Xlist, Ylist, X0, loss_type = "regret",
             family = "reg_ld",  intercept = TRUE,
             delta = 0,  verbose = FALSE)

In [ ]:
summary_cgdro_(fit)

Model Summary:
CGDRO Aggregated Weights:

group     |        1        2        3
weight_   |   0.4788   0.4924   0.0288

CGDRO Aggregated Estimators:

index     |        1        2        3        4        5        6
coef_     |   0.0370  -0.1873  -0.0903  -0.0557  -0.0531   0.0704

Confidence Intervals not provided. Run infer_reg_ld() and pass its result via infer=.


In [ ]:
pred <- predict_cgdro_(fit)  # N x 1 vector of predicted values
head(pred)

[1] -0.143457527 -0.102260302 -0.176093814 -0.009233855  0.028112819
[6] -0.128837656